# 🧠 MNIST Neural Network From Scratch

Built using only:
- NumPy ✅
- Linear Algebra ✅
- Calculus & Chain Rule ✅
- Backpropagation ✅

Not using:
- TensorFlow ❌
- PyTorch ❌
- Scikit-Learn Models ❌


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt


## Load Kaggle Digit Recognizer Dataset

In [ ]:
data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
data = np.array(data)

m, n = data.shape

np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n] / 255.0

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255.0

print(X_train.shape)
print(Y_train.shape)


In [ ]:
def init_params():
    W1 = np.random.randn(10, 784) * np.sqrt(2 / 784)
    b1 = np.zeros((10, 1))

    W2 = np.random.randn(10, 10) * np.sqrt(2 / 10)
    b2 = np.zeros((10, 1))

    return W1, b1, W2, b2


def ReLU(Z):
    return np.maximum(0, Z)


def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)


def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)

    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)

    return Z1, A1, Z2, A2


In [ ]:
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T


def deriv_ReLU(Z):
    return Z > 0


def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size

    one_hot_Y = one_hot(Y)

    dZ2 = A2 - one_hot_Y
    dW2 = (1 / m) * dZ2.dot(A1.T)
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = (1 / m) * dZ1.dot(X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2


In [ ]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 -= alpha * dW1
    b1 -= alpha * db1

    W2 -= alpha * dW2
    b2 -= alpha * db2

    return W1, b1, W2, b2


def get_predictions(A2):
    return np.argmax(A2, axis=0)


def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size


In [ ]:
def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()

    for i in range(iterations):

        Z1, A1, Z2, A2 = forward_prop(
            W1, b1, W2, b2, X
        )

        dW1, db1, dW2, db2 = back_prop(
            Z1, A1, Z2, A2, W2, X, Y
        )

        W1, b1, W2, b2 = update_params(
            W1, b1, W2, b2,
            dW1, db1, dW2, db2,
            alpha
        )

        if i % 50 == 0:
            print(
                f"Iteration {i}: Accuracy = {get_accuracy(get_predictions(A2), Y):.4f}"
            )

    return W1, b1, W2, b2


In [ ]:
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    return get_predictions(A2)


def test_prediction(index, W1, b1, W2, b2):
    current_image = X_train[:, index, None]

    prediction = make_predictions(
        current_image,
        W1, b1, W2, b2
    )

    label = Y_train[index]

    print("Prediction:", prediction)
    print("Label:", label)

    current_image = current_image.reshape((28, 28)) * 255

    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()


In [ ]:
W1, b1, W2, b2 = gradient_descent(
    X_train,
    Y_train,
    iterations=500,
    alpha=0.1
)


In [ ]:
dev_predictions = make_predictions(
    X_dev,
    W1,
    b1,
    W2,
    b2
)

print("Dev Accuracy:", get_accuracy(dev_predictions, Y_dev))


In [ ]:
np.save("W1.npy", W1)
np.save("b1.npy", b1)
np.save("W2.npy", W2)
np.save("b2.npy", b2)

print("Model weights saved successfully!")


In [ ]:
test_prediction(9, W1, b1, W2, b2)
